# reg_with_dummy

> 对应代码：`EconometricsCode/code_in_notes/Chap.1/reg_with_dummy.do`（本 notebook 中的 Stata 代码与 do 文件大体一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.1`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [1]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.1")

import stata_setup
stata_setup.config("/opt/Stata", "mp")


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Single-user 16-core  perpetual
Serial number: 501806307019
  Licensed to: Keep calm and DO your research
               Jichun Si

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. More than 2 billion observations are allowed; see help obs_advice.
      3. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


读入个人数据并构造年收入；再由原始性别编码 a2003（1=男，2=女）变换出 0/1 虚拟变量 gender（0=男，1=女），使其可以直接作为回归中的虚拟变量使用。

In [2]:
%%stata
use ../datasets/chfs2017_ind.dta, clear
gen p_income = a3109*12
gen gender = 2-a2003


. use ../datasets/chfs2017_ind.dta, clear
(chfs2017 ind. data, add demograhic info)

. gen p_income = a3109*12
(93,529 missing values generated)

. gen gender = 2-a2003
(8 missing values generated)

. 


用 bysort 分组调用 outreg2 的 sum 模式，按性别分别输出年收入的样本量与均值，先对两组的收入差异有一个描述性的直观印象。

In [3]:
%%stata
bysort gender: su p_income


-------------------------------------------------------------------------------
-> gender = 0

    Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
    p_income |     13,688    38811.26    34832.48          0     960000

-------------------------------------------------------------------------------
-> gender = 1

    Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
    p_income |     19,793     48108.3    45911.99          0    2400000

-------------------------------------------------------------------------------
-> gender = .

    Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
    p_income |          2       36000    16970.56      24000      48000



以单个虚拟变量 gender 回归年收入：截距给出基准组（男性）的平均收入，gender 的系数给出女性与男性平均收入之差。这再次体现"虚拟变量回归=分组均值比较"，其系数与两样本均值差的 t 检验完全对应。

In [4]:
%%stata
reg p_income gender
outreg2 using reg_with_dummy.tex, replace


. reg p_income gender

      Source |       SS           df       MS      Number of obs   =    33,481
-------------+----------------------------------   F(1, 33479)     =    401.47
       Model |  6.9943e+11         1  6.9943e+11   Prob > F        =    0.0000
    Residual |  5.8326e+13    33,479  1.7422e+09   R-squared       =    0.0118
-------------+----------------------------------   Adj R-squared   =    0.0118
       Total |  5.9026e+13    33,480  1.7630e+09   Root MSE        =     41739



------------------------------------------------------------------------------
    p_income | Coefficient  Std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
      gender |   9297.037   464.0012    20.04   0.000     8387.579     10206.5
       _cons |   38811.26   356.7596   108.79   0.000        38112    39510.53
------------------------------------------------------------------------------

. outreg2 using reg_with_dummy.tex, replace
reg_with_dummy.tex
dir : seeout

. 
